# Nutrition head on Nutrition5k

Stage two. The backbone arrives from Food-101 already knowing what food looks like; this
fits a quantile regression head on 2,755 dishes to predict energy, protein, fat,
carbohydrate, and mass.

Predictions are **intervals, not point estimates**. Portion size cannot be recovered from a
single photograph, so a point estimate would be precision the model does not have. Pinball
loss at the 0.05 / 0.50 / 0.95 quantiles produces the displayed range directly and makes
**interval coverage** a reportable metric: does the stated 90% range actually contain the
truth 90% of the time?

The notebook is deliberately thin. All the logic lives in the `platevision` package, which
is unit tested in CI. Notebooks are a bad place to keep code you intend to trust.

## Before running

**Your Kaggle account must be phone-verified.** GPU access and internet access are both
gated behind it, and an unverified account gets no error saying so: the accelerator setting
is silently ignored and network calls fail to resolve.

In [ ]:
!git clone --depth 1 https://github.com/simonkundrik/plate-vision.git /kaggle/temp/plate-vision
# Path is quoted: an unquoted [extras] suffix is glob-expanded by some shells and
# resolves to nothing, which then fails in a way that reads like a missing package.
%pip install -q -e "/kaggle/temp/plate-vision/model[train,export]"

import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 before running.")

In [ ]:
import subprocess, sys


def run(*args):
    """Run a step and stop the notebook if it fails.

    Jupyter's `!cmd` swallows the exit code, so a training run that dies on its first
    batch produces a kernel that reports success with no artifacts. This turns that
    into an exception, which Kaggle reports as an error.
    """
    printable = ' '.join(str(a) for a in args)
    print(f'$ {printable}', flush=True)
    completed = subprocess.run([sys.executable, *[str(a) for a in args]])
    if completed.returncode != 0:
        raise RuntimeError(f'step failed with exit code {completed.returncode}: {printable}')


## 1. Fetch Nutrition5k

Overhead RGB and metadata only, roughly 1.35 GB. The full archive is 181 GB, almost all of
it rotating side-angle video this project does not use.

**Depth is skipped deliberately.** Nutrition5k ships RGB-D and the original paper shows
depth improves accuracy substantially, but a phone camera has no depth sensor. Training on
it would open a gap between training and deployment that no amount of tuning closes, so the
results here will be worse than the paper's headline numbers as a direct consequence of a
choice made on purpose.

In [ ]:
%cd /kaggle/temp/plate-vision/model
run(
    'data/download_nutrition5k.py',
    '--out',
    '/kaggle/temp/nutrition5k',
    '--workers',
    '16',
    '--depth',
)

## 2. Fetch the Food-101 backbone

The published classifier checkpoint, 86.12% top-1. Pulled from the release rather than
retrained here: stage one took 4.19 hours and its result is already recorded.

Fetching it from a versioned release rather than a Kaggle dataset means this notebook
reproduces from the repository alone, without depending on files in one person's account.

In [ ]:
!mkdir -p /kaggle/temp/backbone
!curl -sL -o /kaggle/temp/backbone/food101-classifier-b0.pt \
  https://github.com/simonkundrik/plate-vision/releases/download/model-v0.1.0/food101-classifier-b0.pt

import hashlib
import pathlib

blob = pathlib.Path("/kaggle/temp/backbone/food101-classifier-b0.pt").read_bytes()
# A truncated download would load as a smaller state dict and transfer fewer weights, and
# the run would look like a bad backbone rather than a bad file.
print(len(blob), "bytes | sha256", hashlib.sha256(blob).hexdigest()[:12])
assert len(blob) == 16847919, "checkpoint download is incomplete"

## 3. Train

Two auxiliary heads are added on top of the phase B recipe, both aimed at calorie
*density* rather than mass. An oracle decomposition put mass error at 14.2% and density
error at 12.8%, roughly independent and combining in quadrature, and the depth work that
was meant to attack mass did not survive measurement.

- **Ingredient supervision.** 133 ingredients appearing in at least 20 dishes. Olive oil
  alone is in half the dataset at 884 kcal/100g, so its presence is real signal about
  kcal per gram rather than seasoning trivia.
- **Classifier distillation.** Dish identity predicts density strongly, and fine-tuning on
  2,424 cafeteria trays erodes the semantics the backbone arrived with. The frozen
  Food-101 classifier scores the same augmented image the student sees.

Neither head is exported. Both exist to shape the features the quantile head reads.

**The weights are small on purpose.** Raw KL against a 101-class teacher runs about 15x the
pinball loss and raw BCE over 133 ingredients about 2.6x, so the first attempt at 0.5 and 0.3
left the regression task at 12% of the gradient. Calorie error rose from 56.7 to 68.8 kcal and
nothing in the loss curve said so, because the total was falling the whole time. 0.010 and
0.057 put each auxiliary term near 15% of the pinball loss.

The trainer prints the balance on the first batch before it trains on it. Read that table
rather than trusting these numbers.

Watch three numbers rather than one:

- **MAE** on energy. Phase B was 56.7 kcal.
- **Coverage** of the 90% interval. Phase B reached 82.2%, and conformal took it to 90.5%.
- **Crossing rate**, how often the 5th percentile exceeds the 95th.

Expect a modest move at best. Density is 12.8% of a 19.4% total combining in quadrature,
so even a large win there shifts the headline by a couple of points.


In [ ]:
run(
    'scripts/train_nutrition.py',
    '--data-root',
    '/kaggle/temp/nutrition5k',
    '--backbone-from',
    '/kaggle/temp/backbone/food101-classifier-b0.pt',
    '--out',
    '/kaggle/working/runs/nutrition',
    '--epochs',
    '60',
    '--batch-size',
    '32',
    '--workers',
    '4',
    '--amp',
    '--mixup-alpha',
    '0.2',
    '--cutmix-alpha',
    '1.0',
    '--mix-prob',
    '0.5',
    '--ema',
    '--ema-decay',
    '0.999',
    '--zoom-out',
    '1.6',
    '--select-on',
    'pinball',
    '--ingredient-weight',
    '0.057',
    '--kd-weight',
    '0.010',
    '--depth',
)

## 4. Evaluate

Coverage calibration is the chart worth having. MAE says how wrong the central estimate is;
calibration says whether the stated uncertainty is honest, which is the claim this project
actually makes.

In [ ]:
run(
    'scripts/evaluate_nutrition.py',
    '--data-root',
    '/kaggle/temp/nutrition5k',
    '--checkpoint',
    '/kaggle/working/runs/nutrition/best.pt',
    '--out',
    '/kaggle/working/runs/nutrition/report',
)

## 5. What came back

Printed so the run's own output records the numbers, rather than them living only in a
fetched artifact.

In [ ]:
import json
import pathlib

for name in ["history.json", "target_transform.json", "report/report.json"]:
    path = pathlib.Path("/kaggle/working/runs/nutrition") / name
    if path.exists():
        print(f"=== {name} ===")
        print(json.dumps(json.loads(path.read_text()), indent=2)[:4000])
        print()
    else:
        print(f"{name} missing")